In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import torch
import numpy as np
import pandas as pd
import json
import time
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from sentence_transformers import SentenceTransformer, util
from google import genai
from dotenv import load_dotenv
from data_loader import load_edos_data, TASK_A_ID2LABEL, TASK_B_ID2LABEL

DATA_DIR='../data/'
RESULTS_DIR='../results/'
MODELS_DIR='../models/'
KG_DIR='../kg/'
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

c:\Users\Ashwin Nair\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
load_dotenv('../.env')
GEMINI_API_KEY=os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found in .env file')

client=genai.Client(api_key=GEMINI_API_KEY) ## Calling the Gemini API
GEMINI_MODEL='gemini-2.5-flash-lite' ## Model Name

time.sleep(3)
response=client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK only.') ## Response from Gemini to see if its working correctly or not
print(f'Gemini API working: {response.text.strip()}')

Gemini API working: OK


In [3]:
def load_roberta(task: str): ## Load fine-tuned RoBERTa model saved from Notebook 02.
    model_path=os.path.join(MODELS_DIR, f'roberta_task_{task}')
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model not found at {model_path}. Run Notebook 02 first.')
    tokenizer=RobertaTokenizer.from_pretrained('roberta-base') ## Roberta Tokenizer
    model=RobertaForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()
    print(f'Loaded RoBERTa Task {task} from {model_path}')
    return model, tokenizer

roberta_a, tokenizer_a = load_roberta('A') ## Loading Roberta Task A
roberta_b, tokenizer_b = load_roberta('B') ## Loading Roberta Task B

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3638.50it/s]


Loaded RoBERTa Task A from ../models/roberta_task_A


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3067.70it/s]


Loaded RoBERTa Task B from ../models/roberta_task_B


In [4]:
kg_path = os.path.join(KG_DIR, 'sexism_kg_v2.json') ## loading our new knowledge graph with restricted relations
with open(kg_path, 'r') as f:
    kg = json.load(f)

all_triples=kg['triples']
entity_index=kg['entity_index']
relation_index=kg['relation_index']

## Knowledge Graph Summary
print(f'KG v2 loaded:')
print(f'Total triples:{len(all_triples)}')
print(f'Unique entities:{len(entity_index)}')
print(f'Unique relations:{len(relation_index)}')

KG v2 loaded:
Total triples:5256
Unique entities:4994
Unique relations:6


In [5]:
# Category to relations mapping — used for pre-filtering before semantic ranking
CATEGORY_TO_RELATIONS = {
    '1. threats, plans to harm and incitement':['THREATENED_WITH'],
    '2. derogation':['STEREOTYPED_AS', 'FRAMED_AS_INFERIOR'],
    '3. animosity':['EXPRESSED_ANIMOSITY_TOWARDS'],
    '4. prejudiced discussions':['ASSIGNED_TO_ROLE', 'IDEOLOGICALLY_DISCREDITED']
}

MAX_TRIPLES_TO_RETRIEVE=3 ## Number of relevant triplets to retrieve
SIMILARITY_THRESHOLD=0.3  # minimum cosine similarity to retrieve a triple
SEMANTIC_MODEL_NAME='all-MiniLM-L6-v2' # fast, lightweight, good quality

print('Configuration loaded.')
print(f'Semantic model: {SEMANTIC_MODEL_NAME}')
print(f'Similarity threshold: {SIMILARITY_THRESHOLD}')
print(f'Max triples per post: {MAX_TRIPLES_TO_RETRIEVE}')

Configuration loaded.
Semantic model: all-MiniLM-L6-v2
Similarity threshold: 0.3
Max triples per post: 3


In [6]:
print('Loading sentence transformer model')
sem_model=SentenceTransformer(SEMANTIC_MODEL_NAME) ## Loads all-MiniLM-L6-v2 — a lightweight but high quality sentence embedding mode

# Convert each triple to a text representation for embedding
def triple_to_text(triple: dict) -> str: ## Convert a triple dict to a readable sentence for embedding.
    return f"{triple['subject']} {triple['relation'].replace('_', ' ').lower()} {triple['object']}"

# Embed all triples — done once at startup
print(f'Embedding {len(all_triples)} triples')
triple_texts=[triple_to_text(t) for t in all_triples]
triple_embeddings=sem_model.encode(triple_texts,batch_size=256,show_progress_bar=True,convert_to_tensor=True,device=DEVICE) ## Create embeddings

print(f'Triple embeddings shape: {triple_embeddings.shape}')
print('Done.')

# Show example triple texts
print('\nSample triple texts:')
for i in range(3):
    print(f'{triple_texts[i]}')

Loading sentence transformer model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5722.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 5256 triples


Batches: 100%|██████████| 21/21 [00:01<00:00, 14.77it/s]

Triple embeddings shape: torch.Size([5256, 384])
Done.

Sample triple texts:
hot girls expressed animosity towards easy
asian ladies framed as inferior 5/10
asian ladies framed as inferior get laid after spending years pretending
